In [ ]:
import sqlite3
import pandas as pd
from loguru import logger
import os

# Aparte tabellen die we gaan gebruiken
conn1 = sqlite3.connect('BikeToDrive_1_Accessoireverkoop.db')
conn2 = sqlite3.connect('BikeToDrive_2_Fietsverkoop.db')
conn3 = sqlite3.connect('BikeToDrive_3_Onderhoud.db')
conn4 = sqlite3.connect('BikeToDrive_4_Accessoire_Inkoop.db')
conn5 = sqlite3.connect('BikeToDrive_5_Fiets_Inkoop.db')

# Een file waar alles erin staat
conn = sqlite3.connect('SDM.db')

2026-03-26 16:51:51.740 | INFO     | __main__:reset_sdm:20 - Start full refresh...
2026-03-26 16:51:51.750 | SUCCESS  | __main__:reset_sdm:31 - Reset geslaagd! Alle tabellen zijn leeg.
2026-03-26 16:51:51.766 | SUCCESS  | __main__:overzetten_alle_data:70 - Accessoire_Verkoop: 100 rijen geladen
2026-03-26 16:51:51.779 | SUCCESS  | __main__:overzetten_alle_data:70 - Accessoire_Verkoop_Monteur: 10 rijen geladen
2026-03-26 16:51:51.792 | SUCCESS  | __main__:overzetten_alle_data:70 - Accessoire_Verkoop_Leverancier: 5 rijen geladen
2026-03-26 16:51:51.802 | SUCCESS  | __main__:overzetten_alle_data:70 - Accessoire_Verkoop_Accessoire: 10 rijen geladen
2026-03-26 16:51:51.813 | SUCCESS  | __main__:overzetten_alle_data:70 - Accessoire_Verkoop_Filiaal: 4 rijen geladen
2026-03-26 16:51:51.824 | SUCCESS  | __main__:overzetten_alle_data:70 - Accessoire_Verkoop_Klant: 20 rijen geladen
2026-03-26 16:51:51.836 | SUCCESS  | __main__:overzetten_alle_data:70 - Fiets_Verkoop: 150 rijen geladen
2026-03-26 1

In [ ]:
def reset_sdm():
    logger.info("Start full refresh...")
    try:
        cursor = conn.cursor()

        """ Haal alle tabellen op
        We moeten dan per tabel oproepen"""

        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tabellen = []

        # De data wordt binnen de 'tabellen' lijst gestopt.
        for tabel in cursor.fetchall():
            tabellen.append(tabel[0])

        # Op basis van deze tabellen lijst wordt de data binnen het tabel verwijderd.
        for tabel in tabellen:
            cursor.execute(f"DELETE FROM {tabel}")

        conn.commit()
        conn.close()
        print("Reset geslaagd! Alle tabellen zijn leeg.")
        
    except Exception as e:
        print(f"Fout bij reset: {e}")

reset_sdm()

In [ ]:
# Wij maken gebruik van prefix, omdat het anders bij ons niet werkt

def bepaal_prefix(bron_db):
    # haal bestandsnaam eruit
    naam = bron_db.replace("BikeToDrive_", "").replace(".db", "")

    # verwijder nummer bijvoorbeeld 1_, 2_, enz.
    naam = naam.split("_", 1)[1]

    # kleine correcties voor naming
    naam = naam.replace("Accessoireverkoop", "Accessoire_Verkoop")
    naam = naam.replace("Fietsverkoop", "Fietsverkoop")
    naam = naam.replace("Accessoire_inkoop", "Accessoire_Inkoop")
    naam = naam.replace("Fiets_Inkoop", "Fiets_Inkoop")

    return naam

In [ ]:
# Nadat de Full refresh uitgevoerd is, maken we gebruik van ''
def overzetten_alle_data():
    sdm_pad = 'SDM.db'
    logger.info("Start schema-driven bulk loading...")

    # De volledige lijst met alle bronnen en doeltabellen
    prefix_map = {
        'BikeToDrive_1_Accessoireverkoop.db': 'Accessoire_Verkoop',
        'BikeToDrive_2_Fietsverkoop.db': 'Fietsverkoop',
        'BikeToDrive_3_Onderhoud.db': 'Onderhoud',
        'BikeToDrive_4_Accessoire_inkoop.db': 'Accessoire_Inkoop',
        'BikeToDrive_5_Fiets_Inkoop.db': 'Fiets_Inkoop'
    }

    conn_sdm = sqlite3.connect(sdm_pad)

      # Loop over databases
    for bron_db in prefix_map:

        if not os.path.exists(bron_db):
            logger.error(f"Bestand niet gevonden: {bron_db}")
            continue

        prefix = prefix_map[bron_db]

        conn_bron = sqlite3.connect(bron_db)
        cursor = conn_bron.cursor()

        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

        tabellen = []
        for t in cursor.fetchall():
            tabellen.append(t[0])

        # Loop over tabellen
        for tabel in tabellen:
            if tabel == "sqlite_sequence":
                continue
            
            try:
                df = pd.read_sql_query(f"SELECT * FROM {tabel}", conn_bron)

                # We maken gebruik vand de prefix zodat de tabelnamen goed staan
                if "Verkoop" in tabel or "Inkoop" in tabel or tabel == "Onderhoud":
                    doel_tabel = tabel
                else:
                    doel_tabel = f"{prefix}_{tabel}"

                df.to_sql(doel_tabel, conn_sdm, if_exists='replace', index=False)

                logger.success(f"{doel_tabel}: {len(df)} rijen geladen")

            except Exception as e:
                logger.warning(f"Fout bij {tabel}: {e}")
        # Nadat de Schema Bulk Loading uitgevoerd is, wordt de connectie bron_db afgesloten
        conn_bron.close()

    conn_sdm.close()
    logger.success("Schema-driven loading compleet!")

# =============== Hieronder wordt de methode uitgevoerd ===============

overzetten_alle_data()